# Bank Customer Churn & Loan Risk Analysis

**Business Question:** Which customer segments are most likely to churn, and where should the bank prioritize retention efforts to protect revenue?

This notebook orchestrates the modular pipeline in `src/` — each step calls into a dedicated module rather than doing the work inline, so the same logic can be reused, tested, or run as a standalone script (`main.py`).

**Dataset:** 10,000 bank customer records (Churn Modelling dataset) — CreditScore, Geography, Age, Tenure, Balance, Products, Activity Status, Salary, and Churn flag.

In [1]:
import sys
sys.path.append('../src')

from data_loader import load_data, validate_schema
from data_cleaner import clean_pipeline
from feature_engineer import engineer_pipeline
from business_metrics import churn_rate_by_segment, compute_risk_score, revenue_at_risk, top_findings
from eda import run_all_plots
from report_generator import generate_markdown_report

import pandas as pd
pd.set_option('display.max_columns', None)

## 1. Load & Validate

In [2]:
df = load_data('../data/raw/bank_churn.csv')
df.head()

[data_loader] Loaded 10000 rows, 13 columns from ../data/raw/bank_churn.csv


,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,Num Of Products,Has Credit Card,Is Active Member,Estimated Salary,Churn
0,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 2. Clean

In [3]:
df = clean_pipeline(df)
df.head()

[data_cleaner] Dropped columns: ['surname']
[data_cleaner] No missing values found.
[data_cleaner] Removed 0 duplicate rows.
[data_cleaner] Flagged 0 outliers in 'balance' (IQR method).
[data_cleaner] Flagged 15 outliers in 'creditscore' (IQR method).


,customerid,creditscore,geography,gender,age,tenure,balance,num_of_products,has_credit_card,is_active_member,estimated_salary,churn,balance_outlier,creditscore_outlier
0,15634602,619,France,Female,42,2,0.00,1,1,1,101348.88,1,False,False
1,15647311,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,False,False
2,15619304,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,False,False
3,15701354,699,France,Female,39,1,0.00,2,0,0,93826.63,0,False,False
4,15737888,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,False,False


## 3. Feature Engineering

In [4]:
df = engineer_pipeline(df)
df[['age', 'age_band', 'tenure', 'tenure_group', 'balance', 'balance_segment', 'value_proxy']].head()

,age,age_band,tenure,tenure_group,balance,balance_segment,value_proxy
0,42,41-50,2,Established (2-4 yrs),0.00,Zero Balance,40539.552
1,41,41-50,1,New (0-1 yrs),83807.86,Mid (50K-120K),95301.748
2,42,41-50,8,Loyal (5+ yrs),159660.80,High (120K+),141369.108
3,39,31-40,1,New (0-1 yrs),0.00,Zero Balance,37530.652
4,43,41-50,2,Established (2-4 yrs),125510.82,High (120K+),106940.132


## 4. Business Metrics

In [5]:
df = compute_risk_score(df)
churn_by_geo = churn_rate_by_segment(df, 'geography')
churn_by_age = churn_rate_by_segment(df, 'age_band')
revenue_risk = revenue_at_risk(df)
findings = top_findings(df)

print('Overall churn rate:', findings['overall_churn_rate_pct'], '%')
churn_by_geo

Overall churn rate: 20.37 %


,geography,customers,churned,churn_rate_pct
0,Germany,2509,814,32.44
1,Spain,2477,413,16.67
2,France,5014,810,16.15


In [6]:
churn_by_age

,age_band,customers,churned,churn_rate_pct
0,51-60,797,448,56.21
1,41-50,2320,788,33.97
2,60+,464,115,24.78
3,31-40,4451,538,12.09
4,18-30,1968,148,7.52


In [7]:
revenue_risk

,risk_tier,customers,total_value_at_risk
0,Medium Risk,4984,"433,147,747"
1,Low Risk,4293,"356,513,213"
2,High Risk,723,"69,615,335"


## 5. Visual Exploration

In [8]:
run_all_plots(df, output_dir='../outputs/charts')

[eda] Saved ../outputs/charts/churn_by_geography.png


[eda] Saved ../outputs/charts/churn_by_age_band.png


[eda] Saved ../outputs/charts/churn_active_vs_inactive.png


[eda] Saved ../outputs/charts/correlation_heatmap.png
[eda] Saved ../outputs/charts/risk_tier_distribution.png


[eda] Saved ../outputs/charts/balance_by_churn.png
[eda] All charts generated.


## 6. Key Findings

- Germany shows the highest churn rate among the three geographies
- The 51-60 age band churns far more than any other segment
- Inactive members churn at roughly 2x the rate of active members
- ~7% of the customer base is flagged High Risk by the rule-based scoring model, but carries a disproportionately smaller share of value at risk than the Medium Risk tier — meaning volume-based prioritization alone would miss where the real exposure sits

## 7. Recommendations

1. Prioritize re-engagement campaigns for inactive members
2. Investigate Germany-specific churn drivers (service, pricing, competition)
3. Cross-sell to single-product customers to raise engagement
4. Build retention offers ranked by value at risk, not just customer count

## 8. Generate Final Report

In [9]:
generate_markdown_report(findings, churn_by_geo, churn_by_age, revenue_risk, '../outputs/summary_report.md')

[report_generator] Report saved to ../outputs/summary_report.md


'# Bank Customer Churn & Risk Report\n_Generated: 2026-08-01 18:59_\n\n## Executive Summary\n\n- **Overall churn rate:** 20.37%\n- **Highest-churn geography:** Germany (32.44% churn)\n- **Highest-churn age band:** 51-60 (56.21% churn)\n- **Inactive members churn at 26.85%** vs **14.27%** for active members\n- **723 customers (7.23% of base)** flagged as High Risk by the rule-based scoring model\n\n## Churn Rate by Geography\n\n| geography   |   customers |   churned |   churn_rate_pct |\n|:------------|------------:|----------:|-----------------:|\n| Germany     |        2509 |       814 |            32.44 |\n| Spain       |        2477 |       413 |            16.67 |\n| France      |        5014 |       810 |            16.15 |\n\n## Churn Rate by Age Band\n\n| age_band   |   customers |   churned |   churn_rate_pct |\n|:-----------|------------:|----------:|-----------------:|\n| 51-60      |         797 |       448 |            56.21 |\n| 41-50      |        2320 |       788 |     